In [1]:
import databento as db
import pandas as pd

In [3]:
client = db.Historical("db-tXmtrDWbbtWvkfudWVLmHrdvYFLii")

In [ ]:
data = client.timeseries.get_range(
    dataset="XNAS.ITCH",
    schema="mbp-10",
    symbols=["CVX", "BAC"],
    start="2026-03-02T13:30:00",
    end="2026-03-31T20:00:00",
)

df = data.to_df()

print(df.shape)
print(df.columns)
print(df.head())

/var/folders/tl/fz2vh8dd3jz7h27v1x_3fbb00000gn/T/ipykernel_23731/762219724.py:1: BentoWarning: The size of this streaming request is greater than 5 GB. It is recommended to submit a batch download request for large volumes of data, or break this request into smaller requests. 
  data = client.timeseries.get_range(


(32169353, 73)
Index(['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side',
       'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence',
       'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00',
       'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01', 'ask_sz_01',
       'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02', 'bid_sz_02',
       'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03', 'ask_px_03',
       'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03', 'bid_px_04',
       'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04', 'ask_ct_04',
       'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05', 'bid_ct_05',
       'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06', 'ask_sz_06',
       'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07', 'bid_sz_07',
       'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08', 'ask_px_08',
       'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08', 'bid_px_09',
       'ask_px_09', 'bid_sz_09', 'ask_sz_09',

In [2]:
df=pd.read_csv("2026-03_BAC_CVX_mbp10.csv")

array([['2026-03-02 13:30:11.478488862+00:00', 10, 2, ..., 3, 1, 'BAC'],
       ['2026-03-02 13:30:11.478533607+00:00', 10, 2, ..., 3, 1, 'BAC'],
       ['2026-03-02 13:30:28.459770941+00:00', 10, 2, ..., 8, 1, 'CVX'],
       ...,
       ['2026-03-31 19:59:59.942335477+00:00', 10, 2, ..., 2, 1, 'BAC'],
       ['2026-03-31 19:59:59.947323486+00:00', 10, 2, ..., 2, 1, 'BAC'],
       ['2026-03-31 19:59:59.998362357+00:00', 10, 2, ..., 1, 2, 'CVX']],
      shape=(32169353, 73), dtype=object)

In [36]:
df["mid"] = (df["bid_px_00"] + df["ask_px_00"]) / 2

df["spread"] = df["ask_px_00"] - df["bid_px_00"]

df["imbalance"] = (
    df["bid_sz_00"] /
    (df["bid_sz_00"] + df["ask_sz_00"])
)

In [40]:
df["ts_event"] = pd.to_datetime(df["ts_event"], utc=True)
df["ts_ny"] = df["ts_event"].dt.tz_convert("America/New_York")

t = df["ts_ny"].dt.time

df = df[
    (t >= pd.Timestamp("09:30:00").time()) &
    (t < pd.Timestamp("16:00:00").time())
].copy()

df = df[
    (df["bid_px_00"] > 0) &
    (df["ask_px_00"] > 0) &
    (df["bid_sz_00"] > 0) &
    (df["ask_sz_00"] > 0)
].copy()

In [42]:
df.columns

Index(['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side',
       'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence',
       'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00',
       'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01', 'ask_sz_01',
       'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02', 'bid_sz_02',
       'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03', 'ask_px_03',
       'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03', 'bid_px_04',
       'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04', 'ask_ct_04',
       'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05', 'bid_ct_05',
       'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06', 'ask_sz_06',
       'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07', 'bid_sz_07',
       'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08', 'ask_px_08',
       'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08', 'bid_px_09',
       'ask_px_09', 'bid_sz_09', 'ask_sz_09', 'bid_ct_09', '

In [43]:
from scipy.linalg import block_diag
import matplotlib.pyplot as plt
